# Day 35: Random Forest Basics (Regression)
**Author:** Sahil-K-Y  
**Phase:** 3 - Tree Models & SVM  
**Date:** Day 035

---

## 1. Random Forest Architecture & Mechanics

A **Random Forest** is an advanced ensemble classifier/regressor consisting of a large collection of structured decision trees. It builds on the principles of **Bagging** but adds an essential layer of randomness to decorrelate the individual trees.

### The Decoupling Mechanism: Feature Subsampling
In regular Bagging, if one or two features are highly dominant predictors in the dataset, almost all bootstrap trees will select these dominant features for their top splits. This makes the trees highly correlated, meaning their errors will also be highly correlated.
* **Random Forest Innovation:** At each node split in each decision tree, only a random subset of $m$ features is considered for splitting, out of the total $M$ features.
  * For **Classification:** Typically, $m = \sqrt{M}$ features are chosen.
  * For **Regression:** Typically, $m = \frac{M}{3}$ features are chosen.
* **Benefit:** By forcing trees to split on different features, Random Forest creates a highly diverse set of predictors. When their predictions are averaged, the variance drops significantly.


## 2. Regression Predictions in Trees and Forests

* **Tree Level Prediction:** A single regression tree partitions the feature space into rectangular regions. For any input $x$, it falls into a leaf node. The prediction of that leaf is the **mean** of the target values of all training samples falling in that region:
  $$\hat{y}_{tree}(x) = \frac{1}{|N_L|} \sum_{i \in N_L} y_i$$
* **Forest Level Prediction:** The Random Forest Regressor combines the predictions of $T$ trees by averaging them:
  $$\hat{y}_{forest}(x) = \frac{1}{T} \sum_{t=1}^T \hat{y}_{tree, t}(x)$$


## 3. Mathematical Evaluation Metrics for Regression

For evaluating regression models, we use four essential metrics:

1. **Mean Squared Error (MSE):** Measures the average squared difference between actual and predicted values. It heavily penalizes large errors.
   $$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$
2. **Root Mean Squared Error (RMSE):** The square root of MSE. It brings the error scale back to the target variable's original unit.
   $$\text{RMSE} = \sqrt{\text{MSE}}$$
3. **Mean Absolute Error (MAE):** The average absolute difference between actual and predicted values. It is linear and more robust to outliers than MSE.
   $$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$
4. **Coefficient of Determination ($R^2$):** Measures the proportion of variance in the target variable that is predictable from the features.
   $$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$
   * $R^2 = 1.0$: Perfect predictions.
   * $R^2 = 0.0$: Equivalent to a baseline model that always predicts the mean $\bar{y}$.
   * $R^2 < 0.0$: Worse than predicting the mean.


## Exercise 1: Training and Evaluating a Default Random Forest Regressor

We will use the **Diabetes dataset** (a standard regression dataset in scikit-learn). The goal is to predict disease progression after one year based on ten baseline features (Age, Sex, BMI, Blood Pressure, and six blood serum measurements).

### Step 1.1: Load Dataset
Let's import `load_diabetes` and construct a pandas DataFrame.


In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

df_diabetes = pd.DataFrame(X, columns=feature_names)
df_diabetes['Progression'] = y

print(f"Dataset shape: {X.shape}")
print(df_diabetes.head())


### Step 1.2: Train-Test Split
We split the data randomly into train and test sets.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")


### Step 1.3: Train a Default RandomForestRegressor
Let's fit the Random Forest Regressor using default parameters.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)
print("Model training completed.")


### Step 1.4: Make Predictions and Calculate Metrics
Let's evaluate the predictions using our four core metrics.


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = rf_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("===== RandomForestRegressor Performance =====")
print(f"R2 Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")


## Exercise 2: Comparative Study — Single Decision Tree vs. Bagging vs. Random Forest

Let's compare the three models to understand how variance reduction improves prediction accuracy.

### Step 2.1: Initialize the Models
We'll define a dictionary containing all three regressors.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

models = {
    'Single Decision Tree': DecisionTreeRegressor(random_state=42),
    'Bagged Trees (Bagging)': BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42), n_estimators=100, random_state=42, n_jobs=-1),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}
print("Models defined.")


### Step 2.2: Evaluate and Compare
Let's write a loop to fit and score each model.


In [ ]:
results_comparison = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_tr_pred = model.predict(X_train)
    y_te_pred = model.predict(X_test)
    
    r2_train = r2_score(y_train, y_tr_pred)
    r2_test = r2_score(y_test, y_te_pred)
    mae_test = mean_absolute_error(y_test, y_te_pred)
    
    results_comparison.append({
        'Model': name,
        'Train R2': r2_train,
        'Test R2': r2_test,
        'Overfit Gap': r2_train - r2_test,
        'Test MAE': mae_test
    })

df_results = pd.DataFrame(results_comparison)
print(df_results.to_string(index=False))


## Exercise 3: Extraction and Plotting of Feature Importances

We will inspect which clinical factors are most critical in predicting diabetes progression.

### Step 3.1: Retrieve Importances
Let's pull the `feature_importances_` attribute and create a sorted DataFrame.


In [ ]:
importances = rf_reg.feature_importances_

df_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

df_imp['Cumulative'] = df_imp['Importance'].cumsum()
print(df_imp)


### Step 3.2: Plot Feature Importances
Let's render a clean bar chart.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.barplot(x='Importance', y='Feature', data=df_imp, palette='coolwarm')
plt.title('Random Forest Feature Importance - Diabetes Progression', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Clinical Features')
plt.tight_layout()
plt.show()


## Exercise 4: Hyperparameter Grid Sweep Heatmap

We will sweep `n_estimators` and `max_depth` to examine their combined impact on the $R^2$ score.

### Step 4.1: Compute Grid Performance
Let's run a nested loop to record R2 scores over the parameter grid.


In [ ]:
n_estimators_list = [10, 50, 100, 200]
max_depth_list = [2, 4, 6, 8, 10, None]

r2_grid = np.zeros((len(max_depth_list), len(n_estimators_list)))

for i, depth in enumerate(max_depth_list):
    for j, n in enumerate(n_estimators_list):
        model = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        r2_grid[i, j] = r2_score(y_test, model.predict(X_test))

print("Grid sweep completed.")


### Step 4.2: Plot the Heatmap
Let's use Seaborn's `heatmap` to visualize the parameter interactions.


In [ ]:
depth_labels = [str(d) if d is not None else 'None' for d in max_depth_list]

sns.heatmap(r2_grid, annot=True, fmt='.4f', cmap='YlGnBu',
            xticklabels=n_estimators_list, yticklabels=depth_labels)
plt.title('R2 Score Heatmap: max_depth vs. n_estimators', fontsize=14, fontweight='bold')
plt.xlabel('Number of Estimators')
plt.ylabel('Maximum Depth')
plt.show()


## Exercise 5: Prediction Scatter and Error Analysis

Let's visually inspect where our model makes errors.

### Step 5.1: Scatter Plot
Let's plot actual vs. predicted values to verify linear correlation.


In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', color='coral')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.title('Actual vs. Predicted Progression', fontsize=12, fontweight='bold')
plt.xlabel('Actual Progression')
plt.ylabel('Predicted Progression')
plt.legend()
plt.show()


### Step 5.2: Error Distribution (Residuals)
Let's plot the distribution of residuals.


In [ ]:
errors = y_test - y_pred
sns.histplot(errors, kde=True, color='purple', bins=20)
plt.axvline(x=0, color='r', linestyle='--', label='Zero Error')
plt.title('Distribution of Prediction Errors (Residuals)', fontsize=12, fontweight='bold')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Count')
plt.legend()
plt.show()

print(f"Mean Prediction Error: {errors.mean():.4f}")
